In [1]:
# Import dependencies

import os
import numpy as np
import math
from datetime import datetime, date
import matplotlib.pyplot as plt
import rasterio as rio

from functions import *
# from Frameworks import *
import warnings
warnings.filterwarnings("ignore")

# Inputs

### Sample Name

In [58]:
head_dir = "/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/"

data_time = "Airbus_2024Feb23"
# data_time = "Airbus_2024Jan11"
# data_time = "Airbus_2024Jun13"

In [59]:
# Directory containing the tif files:
dir_all = os.path.join(head_dir, data_time)
print(f"Processing directory: {dir_all}")


Processing directory: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23


#### Tif images

ALL tif files in the directory

In [60]:
# Get a list of .tif files in the directory:

tif_files, grouped_tif_files = get_file_list(dir_all)

print(f"Found {len(tif_files)} tif files.")

# Example: Print grouped files
for folder, files in grouped_tif_files.items():
    print(f"Folder: {folder}")
    print(f"Files: {files}\n")


Found 140 tif files.
Folder: NovaSAR_01_51481_grd_240111_222538_HH
Files: ['/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_51481_grd_240111_222538_HH/NovaSAR_01_51481_grd_13_240111_222538_HH_1/image_HH.tif', '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_51481_grd_240111_222538_HH/NovaSAR_01_51481_grd_13_240111_222543_HH_2/image_HH.tif', '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_51481_grd_240111_222538_HH/NovaSAR_01_51481_grd_13_240111_222549_HH_3/image_HH.tif', '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_51481_grd_240111_222538_HH/NovaSAR_01_51481_grd_13_240111_222555_HH_4/image_HH.tif', '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_51481_grd_240111_222538_HH/NovaSAR_01_51481_grd_13_240111_222600_HH_5/image_HH.tif']

Folder: NovaSAR_01_51676_grd_240118_082036_HH
Files: ['/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_51676_grd_240118_082036_HH/NovaSAR_01

# Processing

In [61]:
h= 256  # Half height for patch
w= 256  # Half width for patch

In [115]:
f_ii = 13#12#11#10#9#8#7#6#5#11#12#13#14#15#16#17#18#19#20#21#22#23#24#25#26#41#40#39#38#37#31#30#29#28#27#26#25#4#3#2#1#24#23#22#18#17#16#15#14#12#11#10#0#8#7#1#5#3#4#2#32#31#30#29#28#27#33#36#35#34#32#21 #20#13#6#5#0#9 # Index of the file to process

In [116]:
im_super_folders = list(grouped_tif_files.keys())
im_super_foldersii = im_super_folders[f_ii]
tif_pathii = grouped_tif_files[ im_super_foldersii ]

print(f"Processing the Images in this folder: {im_super_foldersii}\n")
print(f"tif images directory:")
tif_pathii

Processing the Images in this folder: NovaSAR_01_52815_grd_240225_084722_HH

tif images directory:


['/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084722_HH_1/image_HH.tif',
 '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084728_HH_2/image_HH.tif',
 '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084734_HH_3/image_HH.tif',
 '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084739_HH_4/image_HH.tif']

In [117]:
from pathlib import Path


for tii in range( len( tif_pathii ) ):
    # Sample image Full path
    tii_path = Path(tif_pathii[tii][:-12])
    parts = tii_path.parts

    im_name = parts[-1]

    # Find the data acquisition date from the metadata XML file:
    try:
        acqdate0 = f"20{'_'.join(im_name.split('_')[-4:-2])}" # or f"20{tif_files[tii][-31:-18]}" # YYYYMMDD_HHMMSS
        dt       = datetime.strptime(acqdate0, '%Y%m%d_%H%M%S')
        acqdate  = dt.strftime('%d/%m/%Y %H:%M:%S') # DD/MM/YYYY HH:MM:SS
    except:
        acqdate0 = f"20{'_'.join(im_name.split('_')[-3:-1])}" # or f"20{tif_files[tii][-31:-18]}" # YYYYMMDD_HHMMSS
        dt       = datetime.strptime(acqdate0, '%Y%m%d_%H%M%S')
        acqdate  = dt.strftime('%d/%m/%Y %H:%M:%S') # DD/MM/YYYY HH:MM:SS

    print("Raw Data Start Time:", acqdate)

    # AIS data directory:
    ais_name = f"aisdk-{acqdate0[:8][:4]}-{acqdate0[:8][4:6]}-{acqdate0[:8][6:8]}"  # Extract the date part (YYYY-MM-DD)       

    csv_dir = f"{head_dir}AIS_dataset/{ais_name}/{ais_name}.csv" 

    ExtractPatchAndAIS(tii_path=tii_path, AIS_path=csv_dir, h=h, w=w)
    print('')

Raw Data Start Time: 25/02/2024 08:47:22
Sample image (tii) name: NovaSAR_01_52815_grd_13_240225_084722_HH_1


Warning 1: TIFFFetchNormalTag:ASCII value for tag "ImageDescription" contains null byte in value; value incorrectly truncated during reading due to implementation limitations


out_nameii: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084722_HH_1/ship_patches/NovaSAR_01_52815_grd_13_240225_084722_HH_1_patch_1_Tanker.tif
Skipping patch for row 13356, col 10595 due to out of bounds.
Skipping patch for row 3838, col 7957 due to out of bounds.
out_nameii: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084722_HH_1/ship_patches/NovaSAR_01_52815_grd_13_240225_084722_HH_1_patch_2_Tug.tif
out_nameii: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084722_HH_1/ship_patches/NovaSAR_01_52815_grd_13_240225_084722_HH_1_patch_3_Tanker.tif
out_nameii: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084722_HH_1/ship_patches/NovaSAR_01_52815_grd_13_240225_084722_HH_1_patc

Warning 1: TIFFFetchNormalTag:ASCII value for tag "ImageDescription" contains null byte in value; value incorrectly truncated during reading due to implementation limitations


out_nameii: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084728_HH_2/ship_patches/NovaSAR_01_52815_grd_13_240225_084728_HH_2_patch_1_HSC.tif
Skipping patch for row -555, col 5709 due to out of bounds.
Skipping patch for row 9451, col -900 due to out of bounds.
out_nameii: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084728_HH_2/ship_patches/NovaSAR_01_52815_grd_13_240225_084728_HH_2_patch_2_Dredging.tif
Skipping patch for row 14856, col 8364 due to out of bounds.
Skipping patch for row -456, col 6333 due to out of bounds.
Saved the updated AIS data with patch_names column to /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084728_HH_2/AIS.csv

Raw Data Start Time: 25/02/2024 08:47:34
Sample image (tii) name: NovaSAR_01_52815_grd_13_240225_084734_HH_3


Warning 1: TIFFFetchNormalTag:ASCII value for tag "ImageDescription" contains null byte in value; value incorrectly truncated during reading due to implementation limitations


out_nameii: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084734_HH_3/ship_patches/NovaSAR_01_52815_grd_13_240225_084734_HH_3_patch_1_Cargo.tif
out_nameii: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084734_HH_3/ship_patches/NovaSAR_01_52815_grd_13_240225_084734_HH_3_patch_2_Tanker.tif
out_nameii: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084734_HH_3/ship_patches/NovaSAR_01_52815_grd_13_240225_084734_HH_3_patch_3_Pilot.tif
Skipping patch for row 8137, col 9125 due to out of bounds.
out_nameii: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084734_HH_3/ship_patches/NovaSAR_01_52815_grd_13_240225_084734_HH_3_patch_4_Cargo.tif
Skipping patch for row 15889, col 4928 due to o

Warning 1: TIFFFetchNormalTag:ASCII value for tag "ImageDescription" contains null byte in value; value incorrectly truncated during reading due to implementation limitations


out_nameii: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084739_HH_4/ship_patches/NovaSAR_01_52815_grd_13_240225_084739_HH_4_patch_1_Pleasure.tif
out_nameii: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084739_HH_4/ship_patches/NovaSAR_01_52815_grd_13_240225_084739_HH_4_patch_2_Dredging.tif
Skipping patch for row -308, col 4628 due to out of bounds.
out_nameii: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084739_HH_4/ship_patches/NovaSAR_01_52815_grd_13_240225_084739_HH_4_patch_3_Dredging.tif
Skipping patch for row 50, col 4923 due to out of bounds.
out_nameii: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084739_HH_4/ship_patches/NovaSAR_01_52815_grd_13_240225_084739_HH_4

## Inshore/Offshore

In [9]:

import geopandas as gpd

# Path to your polyline shapefile
# shapefile_path = "/mnt/e/AssenSAR/CSP LIDAR Data/natural_earth_vector/10m_physical/ne_10m_coastline.shp"  
shapefile_path = "/mnt/e/AssenSAR/CSP LIDAR Data/natural_earth_vector/10m_physical/ne_10m_land.shp"  
# shapefile_path = "/mnt/e/AssenSAR/CSP LIDAR Data/simplified-water-polygons-split-3857/simplified_water_polygons.shp"  
shapefile_path = "/mnt/e/AssenSAR/CSP LIDAR Data/water-polygons-split-4326/water_polygons.shp"  
# shapefile_path = "/mnt/e/AssenSAR/CSP LIDAR Data/land-polygons-complete-4326/land_polygons.shp"
# shapefile_path = "/mnt/e/AssenSAR/CSP LIDAR Data/land-polygons-AOI-4326/land_polygons_AOI.shp"
# Load the shapefile
gdf = gpd.read_file(shapefile_path)
gdf


,x,y,geometry
0,1,41,"POLYGON ((2.00067 40.9995, 0.99933 40.9995, 0...."
1,-11,-72,"POLYGON ((-10.91746 -70.9995, -10.92245 -71.00..."
2,-11,-72,"POLYGON ((-10.75741 -70.9995, -10.73509 -71.00..."
3,148,-11,"POLYGON ((149.00051 -11.0005, 147.99949 -11.00..."
4,-27,-58,"POLYGON ((-25.99907 -56.9995, -25.99907 -58.00..."
...,...,...,...
53294,175,89,"POLYGON ((176.0573 90, 176.0573 88.9995, 174.9..."
53295,176,89,"POLYGON ((177.0573 90, 177.0573 88.9995, 175.9..."
53296,177,89,"POLYGON ((178.0573 90, 178.0573 88.9995, 176.9..."
53297,178,89,"POLYGON ((179.0573 90, 179.0573 88.9995, 177.9..."


In [118]:
def geodesic_distance_to_polygon(point, polygon):
    import shapely
    from shapely.ops import nearest_points

    # from shapely import LineString, Point, Polygon
    from pyproj import Geod
    """
    Calculate the geodesic distance from a point to a polygon.
    
    Parameters:
    - point: A shapely Point object representing the target point.
    - polygon: A shapely Polygon object representing the polygon.
    
    Returns:
    - distance: The geodesic distance from the point to the polygon boundary.
    """
    # Initialize Geod for geodesic calculations on WGS84 ellipsoid
    geod = Geod(ellps="WGS84")
    
    # dist_deg = shapely.distance(point, polygon) # degrees
    _, p2 = nearest_points(point, polygon.boundary) # Get the nearest point on the polygon boundary to the point
    
    # Calculate the inverse geodesic problem: Returns (azimuth1, azimuth2, distance)
    # _, _, dist_geod = geod.inv( point.x, point.y, point.x + dist_deg/(2**.5), point.y + dist_deg/(2**.5) ) # geodesic distance in meters
    _, _, dist_geod = geod.inv( point.x, point.y, p2.x, p2.y ) # geodesic distance in meters
    
    return dist_geod

In [119]:
import glob
import os

# Define the base directory
base_dir = f"/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/{data_time}/{im_super_foldersii}/"

# Find all "AIS.csv" files in the directory and subdirectories
AIS_path_all = glob.glob(os.path.join(base_dir, "**", "AIS.csv"), recursive=True)
print(f"Found {len(AIS_path_all)} AIS.csv files.")
AIS_path_all

Found 4 AIS.csv files.


['/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084722_HH_1/AIS.csv',
 '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084728_HH_2/AIS.csv',
 '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084734_HH_3/AIS.csv',
 '/mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084739_HH_4/AIS.csv']

In [120]:
import pandas as pd
from shapely.geometry import Point, Polygon

dist_thresh = 500  # meters


for AIS_path in AIS_path_all:
    print(f"Processing AIS data from: {AIS_path}")
    ais_data = pd.read_csv(AIS_path)

    ii=0
    for loni, latii in zip(ais_data['Longitude'], ais_data['Latitude']):
        point = Point(loni, latii)
        dist = []
        for polygonii in gdf['geometry'].iloc():
            dist_geod = geodesic_distance_to_polygon(point, polygonii)
            dist.append(dist_geod)
        min_dist = min(dist)
        ais_data.loc[ii,'Dist_to_land'] = min_dist
        if min_dist < dist_thresh:
            ais_data.loc[ii,'Shoreline'] = 'inshore'
        else:
            ais_data.loc[ii,'Shoreline'] = 'offshore'

        # Print the distances for each point
        # print(f"Point ({loni}, {latii}): Distances to land: {min_dist}")
        ii += 1
    ais_data.to_csv(AIS_path, index=False)
    ais_data['Shoreline']


Processing AIS data from: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084722_HH_1/AIS.csv


Processing AIS data from: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084728_HH_2/AIS.csv
Processing AIS data from: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084734_HH_3/AIS.csv
Processing AIS data from: /mnt/e/AssenSAR/CSP LIDAR Data/NovaSAR/Airbus_2024Feb23/NovaSAR_01_52815_grd_240225_084722_HH/NovaSAR_01_52815_grd_13_240225_084739_HH_4/AIS.csv
